# Evaluate Enhanced Queries (Dense + BM25)

Load pre-enhanced queries and evaluate with both retrievers

In [1]:
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement
!apt-get install -qq openjdk-21-jdk-headless
!pip install -q faiss-cpu pytrec-eval transformers torch datasets bm25s PyStemmer nltk requests
!pip install -q --upgrade pillow
!pip install -q pyserini

Cloning into 'graduation'...
remote: Enumerating objects: 713, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 713 (delta 102), reused 204 (delta 69), pack-reused 468 (from 1)
Receiving objects: 100% (713/713), 31.33 MiB | 19.70 MiB/s, done.
Resolving deltas: 100% (274/274), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /us

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/graduation/arabic-rag-query-enhancement
import os, sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

Mounted at /content/drive
/content/graduation/arabic-rag-query-enhancement


In [2]:
!mkdir -p data/miracl_ar
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
!ln -sf "{drive_base}/bm25s_index" data/miracl_ar/bm25s_index
!ln -sf "{drive_base}/corpus_ids.pkl" data/miracl_ar/corpus_ids.pkl

In [3]:
from src.utils.data_loader import MIRACLDataLoader
from src.retrievers.dense import mDPRRetriever
from src.retrievers.bm25 import BM25SRetriever
from src.evaluation.metrics import RetrievalEvaluator, print_metrics
import torch, pickle, gc
from tqdm.notebook import tqdm

## Upload Enhanced Queries File

Upload your `enhanced_queries_exp003.pkl` file using the file upload button below

In [4]:
from google.colab import files
uploaded = files.upload()
print('File uploaded successfully!')

Saving exp_013_csqe_aya_8b.pkl to exp_013_csqe_aya_8b.pkl
File uploaded successfully!


In [5]:
with open('exp_013_csqe_aya_8b.pkl', 'rb') as f:
    data = pickle.load(f)
query_ids = data['query_ids']
query_texts = data['original']
enhanced_queries = data['enhanced']
print(f'Loaded {len(enhanced_queries)} enhanced queries')
print(f'Sample: {enhanced_queries[0][:100]}...')

Loaded 2896 enhanced queries
Sample: من هو علي بن محمد السمري؟ من هو علي بن محمد السمري؟ من هو علي بن محمد السمري؟ من هو علي بن محمد السم...


In [6]:
data_loader = MIRACLDataLoader(language='ar', split='dev')
topics, qrels = data_loader.load_all()
evaluator = RetrievalEvaluator(qrels)
print('Data loaded')

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries
Data loaded


## Dense Evaluation

In [7]:
print('Initializing Dense retriever...')
dense_retriever = mDPRRetriever()
print('Running Dense retrieval...')
dense_results_list = dense_retriever.search(enhanced_queries, k=100, show_progress=True)
dense_results = {}
for i, qid in enumerate(tqdm(query_ids, desc='Formatting')):
    dense_results[qid] = {docid: score for docid, score in dense_results_list[i]}
dense_metrics = evaluator.evaluate(dense_results)
print_metrics(dense_metrics, 'Dense + Query2Doc')
del dense_retriever
torch.cuda.empty_cache()
gc.collect()

Initializing Dense retriever...
Using device: cuda
Loading encoder: castorini/mdpr-tied-pft-msmarco


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/407 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Encoding queries: 100%|██████████| 46/46 [01:31<00:00,  1.99s/it]


Searching FAISS index...


Formatting:   0%|          | 0/2896 [00:00<?, ?it/s]


Dense + Query2Doc
Recall@10      : 0.7073
Recall@100     : 0.8816
NDCG@@10       : 0.5915
MRR            : 0.6225
Queries:    2896


77

## BM25 Evaluation

In [8]:
print('Initializing BM25 retriever...')
bm25_retriever = BM25SRetriever(
    index_path='data/miracl_ar/bm25s_index',
    corpus_ids_path='data/miracl_ar/corpus_ids.pkl'
)
print('Running BM25 retrieval...')
bm25_results_list = bm25_retriever.search(enhanced_queries, k=100, show_progress=True)
bm25_results = {}
for i, qid in enumerate(tqdm(query_ids, desc='Formatting')):
    bm25_results[qid] = {docid: score for docid, score in bm25_results_list[i]}
bm25_metrics = evaluator.evaluate(bm25_results)
print_metrics(bm25_metrics, 'BM25 + Query2Doc')

Initializing BM25 retriever...
Loading BM25S index from data/miracl_ar/bm25s_index...
Loading corpus IDs from data/miracl_ar/corpus_ids.pkl...
✓ Index loaded: 2,061,414 documents
Running BM25 retrieval...
Tokenizing 2896 queries...


Split strings:   0%|          | 0/2896 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/2896 [00:00<?, ?it/s]

Retrieving top 100 documents per query...


BM25S Retrieve:   0%|          | 0/2896 [00:00<?, ?it/s]

Formatting:   0%|          | 0/2896 [00:00<?, ?it/s]


BM25 + Query2Doc
Recall@10      : 0.7447
Recall@100     : 0.9422
NDCG@@10       : 0.6157
MRR            : 0.6380
Queries:    2896


## Comparison

In [9]:
print('\n' + '='*60)
print('RESULTS COMPARISON')
print('='*60)
print(f'\nDense: NDCG@10={dense_metrics["ndcg_cut_10"]:.4f}, Recall@100={dense_metrics["recall_100"]:.4f}, MRR={dense_metrics["recip_rank"]:.4f}')
print(f'BM25:  NDCG@10={bm25_metrics["ndcg_cut_10"]:.4f}, Recall@100={bm25_metrics["recall_100"]:.4f}, MRR={bm25_metrics["recip_rank"]:.4f}')
if dense_metrics['ndcg_cut_10'] > bm25_metrics['ndcg_cut_10']:
    print(f'\nWinner (NDCG@10): Dense')
else:
    print(f'\nWinner (NDCG@10): BM25')
if dense_metrics['recall_100'] > bm25_metrics['recall_100']:
    print(f'Winner (Recall@100): Dense')
else:
    print(f'Winner (Recall@100): BM25')


RESULTS COMPARISON

Dense: NDCG@10=0.5915, Recall@100=0.8816, MRR=0.6225
BM25:  NDCG@10=0.6157, Recall@100=0.9422, MRR=0.6380

Winner (NDCG@10): BM25
Winner (Recall@100): BM25


## Save Results to Google Drive
Save the evaluation metrics and retrieval results to Google Drive for future analysis.

In [10]:
import os
import json
import pickle

# Define the directory to save results in your Drive
results_dir = f'{drive_base}/results'
os.makedirs(results_dir, exist_ok=True)

# Using the uploaded filename as a base for the experiment name
exp_name = 'exp_013_csqe_aya_8b'

# 1. Save metrics as a JSON file
metrics = {
    'dense_metrics': dense_metrics,
    'bm25_metrics': bm25_metrics
}
metrics_path = os.path.join(results_dir, f'{exp_name}_metrics.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)
print(f'Metrics saved to: {metrics_path}')

# 2. Save retrieval results as a pickle file
results = {
    'dense_results': dense_results,
    'bm25_results': bm25_results
}
results_path = os.path.join(results_dir, f'{exp_name}_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(results, f)
print(f'Results saved to: {results_path}')

print('\nAll files successfully saved to Drive!')

Metrics saved to: /content/drive/MyDrive/graduation project/colab_data/results/exp_013_csqe_aya_8b_metrics.json
Results saved to: /content/drive/MyDrive/graduation project/colab_data/results/exp_013_csqe_aya_8b_results.pkl

All files successfully saved to Drive!
